In [219]:
import numpy as np
import pandas as pd

train = pd.read_csv('data/train.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
test = pd.read_csv('data/test.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
events = pd.read_csv('data/events.csv.gz', parse_dates=['event_ts'])

print(train.shape, test.shape, events.shape)
print('доля ботов в train:', train.target.mean().round(4))
train.head()

(11091, 5) (4909, 4) (328905, 14)
доля ботов в train: 0.0811


,cookie_id,cookie_created_at,window_start_ts,window_end_ts,target
0,ck_54a059eb7d3ea68b,2025-11-21 09:30:41,2026-04-06,2026-04-07,0
1,ck_7e4de46eeab82974,2025-09-23 10:10:24,2026-04-06,2026-04-07,0
2,ck_9320229ef6304522,2026-03-04 00:08:02,2026-04-06,2026-04-07,0
3,ck_30ccd25bc1714ed9,2026-04-05 10:52:40,2026-04-06,2026-04-07,0
4,ck_a77c5f05948cdeef,2026-01-01 03:54:35,2026-04-06,2026-04-07,0


## Осмотр и обработка данных

In [220]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 11091 entries, 0 to 11090
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   cookie_id          11091 non-null  str           
 1   cookie_created_at  11091 non-null  datetime64[us]
 2   window_start_ts    11091 non-null  datetime64[us]
 3   window_end_ts      11091 non-null  datetime64[us]
 4   target             11091 non-null  int64         
dtypes: datetime64[us](3), int64(1), str(1)
memory usage: 639.2 KB


In [221]:
print(train.duplicated().sum())
train.isna().sum()

0


cookie_id            0
cookie_created_at    0
window_start_ts      0
window_end_ts        0
target               0
dtype: int64

In [222]:
print(test.duplicated().sum())
test.isna().sum()

0


cookie_id            0
cookie_created_at    0
window_start_ts      0
window_end_ts        0
dtype: int64

In [223]:
print(events.duplicated().sum())
events.isna().sum()

4863


cookie_id             0
event_ts              0
eid                   0
event_name            0
platform              0
user_agent            0
item_id          114597
item_category     32920
item_location     23793
seller_type      133853
search_query     228503
search_page      228503
pointer_x        220365
pointer_y        220365
dtype: int64

In [224]:
events.loc[events.duplicated(keep=False)].sort_values(["cookie_id", "event_ts"])

,cookie_id,event_ts,eid,event_name,platform,user_agent,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y
37283,ck_0027b5d4107d65d8,2026-04-26 05:22:48,300,contact_phone_show,android,Mozilla/5.0 (Linux; Android 14; Redmi Note 11)...,1043647.0,elektronika,krasnoyarsk,private,NaN,NaN,NaN,NaN
141860,ck_0027b5d4107d65d8,2026-04-26 05:22:48,300,contact_phone_show,android,Mozilla/5.0 (Linux; Android 14; Redmi Note 11)...,1043647.0,elektronika,krasnoyarsk,private,NaN,NaN,NaN,NaN
314150,ck_0027ba644f9ae2cd,2026-04-22 17:06:55,200,item_view,ANDROID,Mozilla/5.0 (Linux; Android 11; Pixel 6) Apple...,1036126.0,kvartiry_prodazha,samara,pro,NaN,NaN,NaN,NaN
321075,ck_0027ba644f9ae2cd,2026-04-22 17:06:55,200,item_view,ANDROID,Mozilla/5.0 (Linux; Android 11; Pixel 6) Apple...,1036126.0,kvartiry_prodazha,samara,pro,NaN,NaN,NaN,NaN
10173,ck_0032c08d8987d0ad,2026-04-10 18:13:05,210,photo_swipe,android,Mozilla/5.0 (Linux; Android 14; SM-G991B) Appl...,1051179.0,NaN,moskva,pro,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
182287,ck_fff9fdc2adf5100b,2026-04-23 21:32:38,200,item_view,web,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,1004624.0,elektronika,sankt-peterburg,pro,NaN,NaN,795.0,946.0
148169,ck_fffa4b3ab0203fcf,2026-04-19 16:35:52,100,search_results_view,Android,Mozilla/5.0 (Linux; Android 11; Pixel 6) Apple...,NaN,zhivotnye,ekaterinburg,NaN,котята,3.0,NaN,NaN
328579,ck_fffa4b3ab0203fcf,2026-04-19 16:35:52,100,search_results_view,Android,Mozilla/5.0 (Linux; Android 11; Pixel 6) Apple...,NaN,zhivotnye,ekaterinburg,NaN,котята,3.0,NaN,NaN
91965,ck_fffa4b3ab0203fcf,2026-04-19 22:11:32,200,item_view,ANDROID,Mozilla/5.0 (Linux; Android 11; Pixel 6) Apple...,1043761.0,zhivotnye,sankt-peterburg,pro,NaN,NaN,NaN,NaN


Пока дубли не буду никак обрабатывать, т.к. они это необязательно ошибки, несколько событий могли произойти в одну секунду, что в целом может говорить о действиях бота

Пропуски пока тоже оставлю, они наоборот дают полезную информацию

In [225]:
print(events.event_name.value_counts(), '\n')
print(events.platform.value_counts(), '\n')

event_name
item_view               120817
search_results_view     100402
photo_swipe              36517
favorite_add             19049
seller_page_view         17403
contact_phone_show       11316
captcha_shown             7928
login                     6267
contact_chat_open         6125
contact_message_sent      3081
Name: count, dtype: int64 

platform
desktop    46866
WEB        46476
Web        46365
web        45951
ANDROID    42462
Android    42254
android    42159
iphone      4103
IOS         4100
iOS         4091
ios         4078
Name: count, dtype: int64 



In [226]:
events.platform = events.platform.str.lower().str.strip()
events.platform = events.platform.replace({"iphone": "ios"})

In [227]:
print(events.platform.value_counts(), '\n')

platform
web        138792
android    126875
desktop     46866
ios         16372
Name: count, dtype: int64 



In [228]:
train['cookie_id'].isin(test['cookie_id']).sum()

np.int64(0)

На всякий случай проверил нет ли пересечений

In [229]:
def events_in_window(events, meta):
    ev = events.merge(meta[['cookie_id', 'window_start_ts', 'window_end_ts']], on='cookie_id')
    return ev[(ev.event_ts >= ev.window_start_ts) & (ev.event_ts < ev.window_end_ts)]

ev_tr = events_in_window(events, train)
ev_te = events_in_window(events, test)
print(len(ev_tr), len(ev_te))

198436 89690


In [230]:
train.target.unique()

array([0, 1])

## Создание признаков

Начал с тех признаков, которые основаны на времени 

In [231]:
ev_tr.head()

,cookie_id,event_ts,eid,event_name,platform,user_agent,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y,window_start_ts,window_end_ts
0,ck_d274382b19488771,2026-04-13 12:02:56,200,item_view,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,1048743.0,kvartiry_prodazha,kirov,private,NaN,NaN,675.0,276.0,2026-04-13,2026-04-14
1,ck_fbbed2ff14944ce9,2026-04-08 06:12:14,100,search_results_view,web,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,NaN,bytovaya_tehnika,novosibirsk,NaN,пылесос dyson,2.0,NaN,NaN,2026-04-08,2026-04-09
2,ck_56cc15c7c634cb9f,2026-04-12 17:16:05,100,search_results_view,web,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,NaN,odezhda,sankt-peterburg,NaN,костюм мужской,2.0,NaN,NaN,2026-04-12,2026-04-13
4,ck_f80d85f038cdacee,2026-04-11 16:08:12,200,item_view,desktop,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,1000845.0,bytovaya_tehnika,sankt-peterburg,private,NaN,NaN,NaN,NaN,2026-04-11,2026-04-12
5,ck_e4f046f721d10f81,2026-04-10 16:50:05,200,item_view,ios,Mozilla/5.0 (iPhone; CPU iPhone OS 15_2 like M...,1055188.0,telefony,ekaterinburg,pro,NaN,NaN,NaN,NaN,2026-04-10,2026-04-11


In [232]:
ck_d2 = ev_tr[ev_tr['cookie_id'] == 'ck_d274382b19488771'].sort_values('event_ts')
ck_d2

,cookie_id,event_ts,eid,event_name,platform,user_agent,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y,window_start_ts,window_end_ts
210846,ck_d274382b19488771,2026-04-13 01:09:10,100,search_results_view,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,NaN,kvartiry_prodazha,kaliningrad,NaN,апартаменты купить,5.0,900.0,594.0,2026-04-13,2026-04-14
2490,ck_d274382b19488771,2026-04-13 01:09:43,200,item_view,desktop,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,1026701.0,kvartiry_prodazha,saratov,pro,NaN,NaN,1894.0,764.0,2026-04-13,2026-04-14
221713,ck_d274382b19488771,2026-04-13 03:01:22,300,contact_phone_show,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,1018797.0,kvartiry_prodazha,kaliningrad,NaN,NaN,NaN,1552.0,682.0,2026-04-13,2026-04-14
34365,ck_d274382b19488771,2026-04-13 03:01:27,100,search_results_view,desktop,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,NaN,kvartiry_prodazha,vladivostok,NaN,вторичка 2к,3.0,1139.0,874.0,2026-04-13,2026-04-14
153134,ck_d274382b19488771,2026-04-13 03:01:49,200,item_view,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,1027922.0,NaN,tomsk,private,NaN,NaN,5.0,628.0,2026-04-13,2026-04-14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99273,ck_d274382b19488771,2026-04-13 23:31:49,400,favorite_add,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,1011028.0,kvartiry_prodazha,barnaul,private,NaN,NaN,930.0,535.0,2026-04-13,2026-04-14
188705,ck_d274382b19488771,2026-04-13 23:32:04,200,item_view,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,1049985.0,kvartiry_prodazha,perm,pro,NaN,NaN,1323.0,253.0,2026-04-13,2026-04-14
88442,ck_d274382b19488771,2026-04-13 23:32:13,210,photo_swipe,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,1007550.0,kvartiry_prodazha,ufa,private,NaN,NaN,594.0,360.0,2026-04-13,2026-04-14
78089,ck_d274382b19488771,2026-04-13 23:32:28,300,contact_phone_show,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,1008673.0,kvartiry_prodazha,rostov-na-donu,private,NaN,NaN,306.0,618.0,2026-04-13,2026-04-14


In [233]:
print(ck_d2['event_ts'].max())
print(ck_d2['event_ts'].min())

2026-04-13 23:32:54
2026-04-13 01:09:10


In [234]:
(ck_d2['event_ts'].max() - ck_d2['event_ts'].min()).total_seconds()/60


1343.7333333333333

In [235]:
ck_d2['event_ts'].max() - ck_d2['event_ts'].min()

Timedelta('0 days 22:23:44')

In [236]:
actions = events.groupby("cookie_id").agg(
        n_events=("event_ts", "size"),
        first_event=("event_ts", "min"),
        last_event=("event_ts", "max")).reset_index()
actions.head()

,cookie_id,n_events,first_event,last_event
0,ck_00013fdfa0bd37dd,15,2026-04-24 18:41:16,2026-04-24 20:48:46
1,ck_000c95f1408dcb00,22,2026-04-19 14:49:54,2026-04-20 01:12:32
2,ck_000e8c52636e3bec,34,2026-04-07 06:26:00,2026-04-07 08:17:21
3,ck_0010e31baa4a1fb7,16,2026-04-15 11:53:18,2026-04-15 23:39:34
4,ck_0010ec3874fb5378,74,2026-04-06 02:08:23,2026-04-06 17:59:27


In [237]:
actions["activity_per_minute"] = actions['n_events'] /((actions["last_event"] - actions["first_event"]).dt.total_seconds()/60)
actions.head()

,cookie_id,n_events,first_event,last_event,activity_per_minute
0,ck_00013fdfa0bd37dd,15,2026-04-24 18:41:16,2026-04-24 20:48:46,0.117647
1,ck_000c95f1408dcb00,22,2026-04-19 14:49:54,2026-04-20 01:12:32,0.035334
2,ck_000e8c52636e3bec,34,2026-04-07 06:26:00,2026-04-07 08:17:21,0.305344
3,ck_0010e31baa4a1fb7,16,2026-04-15 11:53:18,2026-04-15 23:39:34,0.022654
4,ck_0010ec3874fb5378,74,2026-04-06 02:08:23,2026-04-06 17:59:27,0.077807


In [238]:
features = actions.copy().drop(['first_event', 'last_event'], axis=1)
features.head()

,cookie_id,n_events,activity_per_minute
0,ck_00013fdfa0bd37dd,15,0.117647
1,ck_000c95f1408dcb00,22,0.035334
2,ck_000e8c52636e3bec,34,0.305344
3,ck_0010e31baa4a1fb7,16,0.022654
4,ck_0010ec3874fb5378,74,0.077807


In [239]:
def features_merge(features, df):
    new_df = features.merge(df, on='cookie_id')
    return new_df

### 2 признак

In [240]:
events.groupby("cookie_id").agg(
        n_events=("event_ts", "size"),
        first_event=("event_ts", "min"),
        last_event=("event_ts", "max")).reset_index()

,cookie_id,n_events,first_event,last_event
0,ck_00013fdfa0bd37dd,15,2026-04-24 18:41:16,2026-04-24 20:48:46
1,ck_000c95f1408dcb00,22,2026-04-19 14:49:54,2026-04-20 01:12:32
2,ck_000e8c52636e3bec,34,2026-04-07 06:26:00,2026-04-07 08:17:21
3,ck_0010e31baa4a1fb7,16,2026-04-15 11:53:18,2026-04-15 23:39:34
4,ck_0010ec3874fb5378,74,2026-04-06 02:08:23,2026-04-06 17:59:27
...,...,...,...,...
15995,ck_ffee36a37e2532f5,81,2026-04-10 11:07:35,2026-04-10 23:21:57
15996,ck_fff88a7920fd0fb6,16,2026-04-17 04:54:29,2026-04-18 00:51:23
15997,ck_fff9fdc2adf5100b,37,2026-04-23 16:10:37,2026-04-23 22:27:33
15998,ck_fffa0622344a36ab,11,2026-04-26 23:05:22,2026-04-26 23:21:25


In [241]:
events_sort = events.sort_values(['cookie_id', 'event_ts'])

In [242]:
events_sort["time_diff"] = events_sort.groupby("cookie_id")["event_ts"].diff().dt.total_seconds()/60
events_sort.head()

,cookie_id,event_ts,eid,event_name,platform,user_agent,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y,time_diff
67286,ck_00013fdfa0bd37dd,2026-04-24 18:41:16,210,photo_swipe,android,Mozilla/5.0 (Linux; Android 12; Pixel 6) Apple...,1012349.0,rabota,moskva,private,NaN,NaN,NaN,NaN,NaN
72380,ck_00013fdfa0bd37dd,2026-04-24 19:16:53,200,item_view,android,Mozilla/5.0 (Linux; Android 12; Pixel 6) Apple...,1054355.0,rabota,moskva,private,NaN,NaN,NaN,NaN,35.616667
189910,ck_00013fdfa0bd37dd,2026-04-24 19:16:54,210,photo_swipe,android,Mozilla/5.0 (Linux; Android 12; Pixel 6) Apple...,1015683.0,detskie_tovary,habarovsk,pro,NaN,NaN,NaN,NaN,0.016667
104718,ck_00013fdfa0bd37dd,2026-04-24 19:17:05,200,item_view,android,Mozilla/5.0 (Linux; Android 12; Pixel 6) Apple...,1028622.0,avtomobili,moskva,private,NaN,NaN,NaN,NaN,0.183333
237561,ck_00013fdfa0bd37dd,2026-04-24 20:05:05,500,login,android,Mozilla/5.0 (Linux; Android 12; Pixel 6) Apple...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,48.000000


In [243]:
time_diff = events_sort.groupby("cookie_id").agg(
    mean_time_diff=("time_diff", "sum"),
    std_time_diff=("time_diff", "std"),
    min_time_diff=("time_diff", "min"),
    max_time_diff=("time_diff", "max"))
time_diff.head()

,mean_time_diff,std_time_diff,min_time_diff,max_time_diff
cookie_id,,,,
ck_00013fdfa0bd37dd,127.500000,15.610213,0.016667,48.000000
ck_000c95f1408dcb00,622.633333,45.951125,0.283333,128.883333
ck_000e8c52636e3bec,111.350000,12.905100,0.000000,70.950000
ck_0010e31baa4a1fb7,706.266667,51.795114,0.166667,137.383333
ck_0010ec3874fb5378,951.066667,33.073415,0.000000,142.783333


In [244]:
features = features_merge(features, time_diff)
features.head()

,cookie_id,n_events,activity_per_minute,mean_time_diff,std_time_diff,min_time_diff,max_time_diff
0,ck_00013fdfa0bd37dd,15,0.117647,127.500000,15.610213,0.016667,48.000000
1,ck_000c95f1408dcb00,22,0.035334,622.633333,45.951125,0.283333,128.883333
2,ck_000e8c52636e3bec,34,0.305344,111.350000,12.905100,0.000000,70.950000
3,ck_0010e31baa4a1fb7,16,0.022654,706.266667,51.795114,0.166667,137.383333
4,ck_0010ec3874fb5378,74,0.077807,951.066667,33.073415,0.000000,142.783333


### 3 признак

здесь перешел к признакам, которые описывают сами действия

In [245]:
events_sort.groupby('cookie_id')['event_name'].agg(list)

cookie_id
ck_00013fdfa0bd37dd    [photo_swipe, item_view, photo_swipe, item_vie...
ck_000c95f1408dcb00    [search_results_view, photo_swipe, item_view, ...
ck_000e8c52636e3bec    [item_view, search_results_view, favorite_add,...
ck_0010e31baa4a1fb7    [item_view, search_results_view, item_view, fa...
ck_0010ec3874fb5378    [favorite_add, favorite_add, item_view, search...
                                             ...                        
ck_ffee36a37e2532f5    [seller_page_view, search_results_view, search...
ck_fff88a7920fd0fb6    [item_view, item_view, item_view, photo_swipe,...
ck_fff9fdc2adf5100b    [item_view, seller_page_view, item_view, item_...
ck_fffa0622344a36ab    [search_results_view, contact_phone_show, item...
ck_fffa4b3ab0203fcf    [item_view, item_view, search_results_view, it...
Name: event_name, Length: 16000, dtype: object

In [246]:
events_sort['previous_event'] = events_sort.groupby('cookie_id')['event_name'].shift(1)
events_sort.head()

,cookie_id,event_ts,eid,event_name,platform,user_agent,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y,time_diff,previous_event
67286,ck_00013fdfa0bd37dd,2026-04-24 18:41:16,210,photo_swipe,android,Mozilla/5.0 (Linux; Android 12; Pixel 6) Apple...,1012349.0,rabota,moskva,private,NaN,NaN,NaN,NaN,NaN,NaN
72380,ck_00013fdfa0bd37dd,2026-04-24 19:16:53,200,item_view,android,Mozilla/5.0 (Linux; Android 12; Pixel 6) Apple...,1054355.0,rabota,moskva,private,NaN,NaN,NaN,NaN,35.616667,photo_swipe
189910,ck_00013fdfa0bd37dd,2026-04-24 19:16:54,210,photo_swipe,android,Mozilla/5.0 (Linux; Android 12; Pixel 6) Apple...,1015683.0,detskie_tovary,habarovsk,pro,NaN,NaN,NaN,NaN,0.016667,item_view
104718,ck_00013fdfa0bd37dd,2026-04-24 19:17:05,200,item_view,android,Mozilla/5.0 (Linux; Android 12; Pixel 6) Apple...,1028622.0,avtomobili,moskva,private,NaN,NaN,NaN,NaN,0.183333,photo_swipe
237561,ck_00013fdfa0bd37dd,2026-04-24 20:05:05,500,login,android,Mozilla/5.0 (Linux; Android 12; Pixel 6) Apple...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,48.000000,item_view


In [247]:
events_sort['same_as_previous'] = events_sort['event_name'] == events_sort['previous_event']

repeat_features = events_sort.groupby('cookie_id').agg(
    repeated_actions=('same_as_previous', 'mean'),
    repeated_actions_ratio=('same_as_previous', 'sum')).reset_index()

repeat_features.head()

,cookie_id,repeated_actions,repeated_actions_ratio
0,ck_00013fdfa0bd37dd,0.066667,1
1,ck_000c95f1408dcb00,0.409091,9
2,ck_000e8c52636e3bec,0.147059,5
3,ck_0010e31baa4a1fb7,0.187500,3
4,ck_0010ec3874fb5378,0.283784,21


In [248]:
features = features_merge(features, repeat_features)
features.head()

,cookie_id,n_events,activity_per_minute,mean_time_diff,std_time_diff,min_time_diff,max_time_diff,repeated_actions,repeated_actions_ratio
0,ck_00013fdfa0bd37dd,15,0.117647,127.500000,15.610213,0.016667,48.000000,0.066667,1
1,ck_000c95f1408dcb00,22,0.035334,622.633333,45.951125,0.283333,128.883333,0.409091,9
2,ck_000e8c52636e3bec,34,0.305344,111.350000,12.905100,0.000000,70.950000,0.147059,5
3,ck_0010e31baa4a1fb7,16,0.022654,706.266667,51.795114,0.166667,137.383333,0.187500,3
4,ck_0010ec3874fb5378,74,0.077807,951.066667,33.073415,0.000000,142.783333,0.283784,21


### 4 признак

In [249]:
proportions = events.groupby('cookie_id')['event_name'].value_counts(normalize=True).reset_index()
proportions.head()

,cookie_id,event_name,proportion
0,ck_00013fdfa0bd37dd,item_view,0.266667
1,ck_00013fdfa0bd37dd,search_results_view,0.200000
2,ck_00013fdfa0bd37dd,photo_swipe,0.200000
3,ck_00013fdfa0bd37dd,contact_chat_open,0.133333
4,ck_00013fdfa0bd37dd,login,0.133333


In [250]:
proportions['entropy_part'] = -proportions['proportion'] * np.log(proportions['proportion'])
proportions.head()

,cookie_id,event_name,proportion,entropy_part
0,ck_00013fdfa0bd37dd,item_view,0.266667,0.352468
1,ck_00013fdfa0bd37dd,search_results_view,0.200000,0.321888
2,ck_00013fdfa0bd37dd,photo_swipe,0.200000,0.321888
3,ck_00013fdfa0bd37dd,contact_chat_open,0.133333,0.268654
4,ck_00013fdfa0bd37dd,login,0.133333,0.268654


In [251]:
event_entropy = proportions.groupby('cookie_id')['entropy_part'].sum().rename('event_entropy').reset_index()
event_entropy.head()


,cookie_id,event_entropy
0,ck_00013fdfa0bd37dd,1.714088
1,ck_000c95f1408dcb00,1.097166
2,ck_000e8c52636e3bec,1.672241
3,ck_0010e31baa4a1fb7,1.488036
4,ck_0010ec3874fb5378,1.521574


In [252]:
features = features_merge(features, event_entropy)
features.head()

,cookie_id,n_events,activity_per_minute,mean_time_diff,std_time_diff,min_time_diff,max_time_diff,repeated_actions,repeated_actions_ratio,event_entropy
0,ck_00013fdfa0bd37dd,15,0.117647,127.500000,15.610213,0.016667,48.000000,0.066667,1,1.714088
1,ck_000c95f1408dcb00,22,0.035334,622.633333,45.951125,0.283333,128.883333,0.409091,9,1.097166
2,ck_000e8c52636e3bec,34,0.305344,111.350000,12.905100,0.000000,70.950000,0.147059,5,1.672241
3,ck_0010e31baa4a1fb7,16,0.022654,706.266667,51.795114,0.166667,137.383333,0.187500,3,1.488036
4,ck_0010ec3874fb5378,74,0.077807,951.066667,33.073415,0.000000,142.783333,0.283784,21,1.521574


### 5 признак

In [253]:
events.head()

,cookie_id,event_ts,eid,event_name,platform,user_agent,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y
0,ck_5efbea1befdefe1b,2026-04-26 09:11:24,200,item_view,desktop,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,1027450.0,elektronika,kaliningrad,pro,NaN,NaN,NaN,NaN
1,ck_c4ca1434f3778f1d,2026-04-20 14:04:31,100,search_results_view,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:1...,NaN,telefony,habarovsk,NaN,iphone 13 128,4.0,NaN,NaN
2,ck_d274382b19488771,2026-04-13 12:02:56,200,item_view,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,1048743.0,kvartiry_prodazha,kirov,private,NaN,NaN,675.0,276.0
3,ck_fbbed2ff14944ce9,2026-04-08 06:12:14,100,search_results_view,web,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,NaN,bytovaya_tehnika,novosibirsk,NaN,пылесос dyson,2.0,NaN,NaN
4,ck_56cc15c7c634cb9f,2026-04-12 17:16:05,100,search_results_view,web,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,NaN,odezhda,sankt-peterburg,NaN,костюм мужской,2.0,NaN,NaN


In [254]:
pointer = events.groupby('cookie_id').agg(
    pointer_count=('pointer_x', 'count'),
    pointer_x_nunique=('pointer_x', 'nunique'),
    pointer_y_nunique=('pointer_y', 'nunique'),
    pointer_x_std=('pointer_x', 'std'),
    pointer_y_std=('pointer_y', 'std'),
    pointer_x_min=('pointer_x', 'min'),
    pointer_x_max=('pointer_x', 'max'),
    pointer_y_min=('pointer_y', 'min'),
    pointer_y_max=('pointer_y', 'max')).reset_index()

pointer.head()

,cookie_id,pointer_count,pointer_x_nunique,pointer_y_nunique,pointer_x_std,pointer_y_std,pointer_x_min,pointer_x_max,pointer_y_min,pointer_y_max
0,ck_00013fdfa0bd37dd,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
1,ck_000c95f1408dcb00,22,22,22,446.190067,344.046700,2.0,1884.0,100.0,1059.0
2,ck_000e8c52636e3bec,31,31,31,561.544146,277.719426,18.0,1910.0,28.0,1048.0
3,ck_0010e31baa4a1fb7,16,16,15,501.890941,361.434157,72.0,1522.0,2.0,1023.0
4,ck_0010ec3874fb5378,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN


In [255]:
features = features_merge(features, pointer)
features.head()

,cookie_id,n_events,activity_per_minute,mean_time_diff,std_time_diff,min_time_diff,max_time_diff,repeated_actions,repeated_actions_ratio,event_entropy,pointer_count,pointer_x_nunique,pointer_y_nunique,pointer_x_std,pointer_y_std,pointer_x_min,pointer_x_max,pointer_y_min,pointer_y_max
0,ck_00013fdfa0bd37dd,15,0.117647,127.500000,15.610213,0.016667,48.000000,0.066667,1,1.714088,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
1,ck_000c95f1408dcb00,22,0.035334,622.633333,45.951125,0.283333,128.883333,0.409091,9,1.097166,22,22,22,446.190067,344.046700,2.0,1884.0,100.0,1059.0
2,ck_000e8c52636e3bec,34,0.305344,111.350000,12.905100,0.000000,70.950000,0.147059,5,1.672241,31,31,31,561.544146,277.719426,18.0,1910.0,28.0,1048.0
3,ck_0010e31baa4a1fb7,16,0.022654,706.266667,51.795114,0.166667,137.383333,0.187500,3,1.488036,16,16,15,501.890941,361.434157,72.0,1522.0,2.0,1023.0
4,ck_0010ec3874fb5378,74,0.077807,951.066667,33.073415,0.000000,142.783333,0.283784,21,1.521574,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
